# 13E continuation — study-area Sentinel-2 tile-union QA and reference design

**Run inside your existing Hakaluki project and registered Earth Engine environment.** This notebook consumes `results/tables/analysis13/13E_sentinel2_point_coverage_QA.csv` and the existing 550 coordinate points. It makes **no automatic claims of class accuracy**.

1. Determine truly usable fixed-point coverage of same-date tile unions, by point ID; do not add single-tile totals.
2. Sample multispectral candidate evidence at valid points for *manual inspection*; do not use SCL water as a reference label.
3. Configure a probability-based stratified independent reference design from a matched SAR raster (optional until suitable image review imagery exists).

In [1]:
from pathlib import Path
import os, sys
import pandas as pd
start=Path(os.environ.get('HAKALUKI_ROOT',Path.cwd())).expanduser().resolve()
for root in [start,*start.parents]:
    if (root/'src/analysis13/core.py').is_file():
        ROOT=root;break
else:
    raise FileNotFoundError('Start Jupyter in your project root, or set HAKALUKI_ROOT')
sys.path.insert(0,str(ROOT))
tables=ROOT/'results/tables/analysis13'
qa=pd.read_csv(tables/'13E_sentinel2_point_coverage_QA.csv')
pts=pd.read_csv(tables/'13A_fixed_reference_points.csv')
print('Root:',ROOT,'candidates:',len(qa),'sample points:',len(pts))
assert pts[['longitude','latitude']].notna().all().all()

Root: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project candidates: 27 sample points: 550


## Same-optical-date tile union coverage

The prior QA reported counts from each tile separately. This obtains SCL labels per point from **all matching same-day tiles** and calculates point-ID union coverage. SCL classes 4, 5 and 6 count as *potentially usable*; they are not manually validated water labels. If tile values disagree, retain the entire class set in the separate point-evidence file.

In [2]:
import ee
project=os.environ.get('EE_PROJECT')
if project:
    ee.Initialize(project=project)
else:
    ee.Initialize()  # Uses project from registered Earth Engine environment
features=[ee.Feature(ee.Geometry.Point([float(r.longitude),float(r.latitude)]),{'point_id':str(i)})
          for i,r in pts.reset_index(drop=True).iterrows()]
fc=ee.FeatureCollection(features)
usable={4,5,6}
all_samples=[]
for rec in qa.itertuples(index=False):
    img=ee.Image(rec.sentinel2_id).select('SCL')
    # sampleRegions returns only pixels valid for this tile, and may drop outside tile bounds.
    sample=img.sampleRegions(collection=fc,scale=20,geometries=False).getInfo()['features']
    for f in sample:
        pr=f['properties'];v=pr.get('SCL');id_=pr.get('point_id')
        if id_ is None or v is None:continue
        all_samples.append({'sar_date':str(rec.sar_date),'optical_date':str(rec.optical_date),
                            'sentinel2_id':rec.sentinel2_id,'point_id':str(id_),
                            'SCL':int(v),'usable_SCL':int(v) in usable,
                            'offset_days':int(rec.offset_days)})
samples=pd.DataFrame(all_samples)
assert not samples.empty,'EE returned no samples'
samples.to_csv(tables/'13E_S2_sampled_point_SCL_all_candidates.csv',index=False)
# Deduplicate by point_id across same-date tiles: usable if any tile has eligible SCL.
point_union=(samples.groupby(['sar_date','optical_date','point_id'],as_index=False)
             .agg(usable_any=('usable_SCL','any'),
                  n_tiles_observed=('sentinel2_id','nunique'),
                  scl_values=('SCL',lambda x:';'.join(map(str,sorted(set(x)))))))
point_union.to_csv(tables/'13E_S2_point_tile_union_evidence.csv',index=False)
summary=(point_union.groupby(['sar_date','optical_date'],as_index=False)
         .agg(covered_unique_points=('point_id','nunique'),
              usable_unique_points=('usable_any','sum'),
              conflicting_tiles=('scl_values',lambda x:sum(';' in v for v in x))))
summary['total_points']=len(pts)
summary['usable_fraction']=summary.usable_unique_points/len(pts)
summary['offset_days']=(pd.to_datetime(summary.optical_date)-pd.to_datetime(summary.sar_date)).dt.days
summary.sort_values(['sar_date','usable_unique_points'],ascending=[True,False],inplace=True)
summary.to_csv(tables/'13E_S2_same_date_tile_union_QA.csv',index=False)
display(summary)
print('Written exact same-day tile union point QA. NOT classification accuracy.')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


,sar_date,optical_date,covered_unique_points,usable_unique_points,conflicting_tiles,total_points,usable_fraction,offset_days
1,2022-05-11,2022-05-08,219,200,0,550,0.363636,-3
0,2022-05-11,2022-05-06,532,36,0,550,0.065455,-5
3,2022-05-23,2022-05-28,255,198,36,550,0.360000,5
2,2022-05-23,2022-05-26,532,11,0,550,0.020000,3
5,2023-06-11,2023-06-10,550,207,85,550,0.376364,-1
4,2023-06-11,2023-06-07,296,0,0,550,0.000000,-4
6,2023-06-23,2023-06-25,532,86,0,550,0.156364,2
7,2023-06-23,2023-06-27,271,25,3,550,0.045455,4
8,2024-06-05,2024-06-01,286,228,24,550,0.414545,-4
9,2024-06-05,2024-06-06,271,0,0,550,0.000000,1


Written exact same-day tile union point QA. NOT classification accuracy.


## Choose a SAR acquisition and make the blind probability sample

This step uses your existing `make_sample` implementation and matched 13B **four-band** GeoTIFF. The existing 550 fixed coordinates should **not** be substituted for a designed validation sample if your intention is area-weighted accuracy inference. Perform the optical reference interpretation **without looking at the SAR prediction or private sample fields**.

Setting the scene ID/raster must be deliberate and must match. The existing code blocks scoring until labels are independently assigned.

In [3]:
import hashlib
from src.analysis13.core import load_scenes
from src.analysis13.validation import make_sample
scenes=load_scenes(ROOT)
cache=ROOT/'data/processed/analysis13_scene_rasters'
print('Cached matched scene rasters:')
for f in sorted(cache.glob('*_matched_30m_EPSG32646.tif')):print(f.name)
# Choose ONE sufficiently optically observable acquisition based on new union coverage.
REFERENCE_SCENE_ID=None  # exact full Sentinel-1 scene_id
REFERENCE_RASTER=None  # absolute path to its 13B cached four-band GeoTIFF
if REFERENCE_SCENE_ID and REFERENCE_RASTER:
    ras=Path(REFERENCE_RASTER)
    token=hashlib.sha256(REFERENCE_SCENE_ID.encode()).hexdigest()[:12]
    assert ras.name.startswith(token),'Scene ID / raster mismatch'
    output=make_sample(ROOT,ras,scene_id=REFERENCE_SCENE_ID,n_each=60)
    print(output)
    print('Fill independently: results/tables/analysis13/13E_BLIND_reference_labelling.csv')
else:
    print('Select an eligible scene/raster after the point-union QA. No labels invented.')

Cached matched scene rasters:
24cc1cfe31de_matched_30m_EPSG32646.tif
38e2a62d908a_matched_30m_EPSG32646.tif
512cef90bade_matched_30m_EPSG32646.tif
51de5867eb20_matched_30m_EPSG32646.tif
6f0d1f3b6ccd_matched_30m_EPSG32646.tif
877454787fe4_matched_30m_EPSG32646.tif
8d6bc5ef08f7_matched_30m_EPSG32646.tif
95b36597a858_matched_30m_EPSG32646.tif
987b817ddf80_matched_30m_EPSG32646.tif
a3b4cb58ef80_matched_30m_EPSG32646.tif
fc350713d26d_matched_30m_EPSG32646.tif
Select an eligible scene/raster after the point-union QA. No labels invented.


## Final gate

The 13E scorer expects complete, independently reviewed reference imagery labels with dates, provenance and analyst. Leave this off until that work exists. Label `unobservable` when interpretation is impossible; do not recode it as `dry_or_other`. If many reference points are unobservable, define a smaller documented estimand and redesigned sampling strategy, rather than claiming full-domain accuracy.

In [4]:
from src.analysis13.validation import score_reference
RUN_REFERENCE_SCORING=True
if RUN_REFERENCE_SCORING:
    print(score_reference(ROOT))
else:
    print('Validation accuracy not computed: independently interpreted blind labels required.')

FileNotFoundError: Need the generated sample metadata and independently labelled CSV